# IMPORT

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

from collections import deque
import random
import math

## 3.4 SỐ teration

In [2]:
## Hàm tính số iteration

def compute_iterations(
        outlier_ratio,
        sample_size,
        p=0.99
):

    e = outlier_ratio
    s = sample_size

    numerator = np.log(1 - p)
    denominator = np.log(
        1 - (1 - e)**s
    )

    N = int(
        np.ceil(
            numerator / denominator
        )
    )

    return N

In [3]:
## Kiểm tra bảng trong PDF

outliers = [0.1, 0.3, 0.5, 0.7]

for e in outliers:

    N = compute_iterations(
        outlier_ratio=e,
        sample_size=2
    )

    print(
        f"Outlier={e*100:.0f}% "
        f"-> N={N}"
    )

Outlier=10% -> N=3
Outlier=30% -> N=7
Outlier=50% -> N=17
Outlier=70% -> N=49


## 3.5 Thực nghiệm

In [4]:
## Kiểm tra RANSAC thành công bao nhiêu %

def experiment(
        outlier_ratio,
        sample_size,
        epsilon,
        trials=100
):

    N = compute_iterations(
        outlier_ratio,
        sample_size
    )

    success = 0

    for _ in range(trials):

        n_out = int(
            100 * outlier_ratio
            / (1 - outlier_ratio)
        )

        points = generate_line_data(
            n_inliers=100,
            n_outliers=n_out
        )

        model, inliers = ransac(
            points,
            fit_line,
            line_distance,
            sample_size,
            epsilon,
            N
        )

        if len(inliers) > 80:
            success += 1

    return success / trials

In [5]:
## Chạy với N lý thuyết

for e in [0.2, 0.4, 0.6]:

    rate = experiment(
        e,
        sample_size=2,
        epsilon=0.6,
        trials=100
    )

    print(
        f"Outlier={e*100:.0f}% "
        f"Success={rate*100:.2f}%"
    )

NameError: name 'generate_line_data' is not defined

In [6]:
## Giảm N xuống

def experiment_with_custom_N(
        outlier_ratio,
        sample_size,
        epsilon,
        N,
        trials=100
):

    success = 0

    for _ in range(trials):

        n_out = int(
            100 * outlier_ratio
            / (1 - outlier_ratio)
        )

        points = generate_line_data(
            100,
            n_out
        )

        model, inliers = ransac(
            points,
            fit_line,
            line_distance,
            sample_size,
            epsilon,
            N
        )

        if len(inliers) > 80:
            success += 1

    return success / trials

In [7]:
## So sánh

e = 0.5

N_theory = compute_iterations(
    e,
    sample_size=2
)

for factor in [
        1.0,
        0.75,
        0.5,
        0.25
]:

    N = int(
        N_theory * factor
    )

    rate = experiment_with_custom_N(
        e,
        sample_size=2,
        epsilon=0.6,
        N=N
    )

    print(
        f"N={N} "
        f"Success={rate*100:.2f}%"
    )

NameError: name 'generate_line_data' is not defined

In [8]:
## Vẽ biểu đồ

Ns = []
rates = []

for factor in np.arange(
        0.2,
        1.2,
        0.1
):

    N = int(
        N_theory * factor
    )

    rate = experiment_with_custom_N(
        e,
        sample_size=2,
        epsilon=0.6,
        N=N
    )

    Ns.append(N)
    rates.append(rate)

plt.plot(
    Ns,
    rates,
    marker='o'
)

plt.xlabel(
    "Number of Iterations"
)

plt.ylabel(
    "Success Rate"
)

plt.title(
    "RANSAC Success vs Iterations"
)

plt.grid(True)

plt.show()

NameError: name 'generate_line_data' is not defined